# Hugging Face Embeddings Demo (MPS on Apple Silicon)

This notebook demonstrates loading a Hugging Face embedding model and generating embeddings with two APIs:
- Raw `transformers` (`AutoTokenizer` + `AutoModel`)
- `sentence-transformers` convenience API

MPS is required by design for this demo (no CPU fallback).

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pip") is None:
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "transformers",
    "torch",
    "sentence-transformers",
])


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


0

In [2]:
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Embeddings map text to dense vectors.",
    "Apple Silicon can accelerate PyTorch workloads with MPS.",
    "This is an encode-only notebook demo.",
]

print(f"Model: {MODEL_ID}")
print(f"Sentence count: {len(sentences)}")

/Users/mtp/git/ai-eng-toolkit/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model: sentence-transformers/all-MiniLM-L6-v2
Sentence count: 4


In [3]:
if not torch.backends.mps.is_available():
    raise RuntimeError(
        "MPS is not available on this machine. This notebook requires Apple Silicon GPU (mps)."
    )

device = torch.device("mps")
print(f"Using device: {device}")

Using device: mps


In [4]:
# Load the model assets directly from Transformers (no SentenceTransformer wrapper).
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device)
model.eval()

# Convert text to padded token IDs + attention mask tensors.
encoded = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    return_tensors="pt",
)
# Move tokenized inputs to MPS so model inference runs on GPU.
encoded = {k: v.to(device) for k, v in encoded.items()}

# Run a forward pass to get per-token hidden states.
with torch.no_grad():
    output = model(**encoded)

# `last_hidden_state` has shape: [batch, seq_len, hidden_dim].
token_embeddings = output.last_hidden_state
# Expand the attention mask so padding tokens can be excluded from pooling.
attention_mask = encoded["attention_mask"].unsqueeze(-1).expand(token_embeddings.size()).float()
masked_embeddings = token_embeddings * attention_mask
# Mean-pool only non-padding token vectors to get one embedding per sentence.
sum_embeddings = masked_embeddings.sum(dim=1)
sum_mask = attention_mask.sum(dim=1).clamp(min=1e-9)
raw_embeddings = sum_embeddings / sum_mask
# L2-normalize sentence vectors, then move back to CPU as NumPy for inspection.
raw_embeddings = torch.nn.functional.normalize(raw_embeddings, p=2, dim=1)
raw_embeddings_cpu = raw_embeddings.detach().cpu().numpy()

print("Raw transformers embedding shape:", raw_embeddings_cpu.shape)
print("First vector (first 8 dims):", np.round(raw_embeddings_cpu[0][:8], 4))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2779.65it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Raw transformers embedding shape: (4, 384)
First vector (first 8 dims): [ 0.0439  0.0589  0.0482  0.0775  0.0267 -0.0376 -0.0026 -0.0599]


In [5]:
st_model = SentenceTransformer(MODEL_ID, device="mps")
st_embeddings = st_model.encode(
    sentences,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

print("SentenceTransformer embedding shape:", st_embeddings.shape)
print("First vector (first 8 dims):", np.round(st_embeddings[0][:8], 4))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2567.05it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SentenceTransformer embedding shape: (4, 384)
First vector (first 8 dims): [ 0.0439  0.0589  0.0482  0.0775  0.0267 -0.0376 -0.0026 -0.0599]


In [6]:
assert raw_embeddings_cpu.shape[0] == len(sentences), "Raw transformers row count mismatch"
assert st_embeddings.shape[0] == len(sentences), "SentenceTransformer row count mismatch"

print(f"Raw embedding dimension: {raw_embeddings_cpu.shape[1]}")
print(f"SentenceTransformer embedding dimension: {st_embeddings.shape[1]}")
print("Validation passed. Minor numeric differences between methods are expected.")

Raw embedding dimension: 384
SentenceTransformer embedding dimension: 384
Validation passed. Minor numeric differences between methods are expected.
